# DSM Univariate Gaussian

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

Clean vs noisy marginal and closed-form denoising score.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/dsm-gaussian/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(142)

# The demo's default noise level: clean data N(0, 1), noisy x~ = x + sigma eps
sigma = 0.5
exact_slope = -1 / (1 + sigma ** 2)            # noisy marginal N(0, 1 + sigma^2): s(x) = -x / (1 + sigma^2)
print(f"closed-form score of the noisy marginal: s(x) = {exact_slope:.4f} x")

# ── Learn s(x) = a x by denoising score matching ────────────────────────────
a = torch.zeros((), requires_grad=True)
opt = torch.optim.Adam([a], lr=0.05)
for step in range(1, 1501):
    x = torch.randn(4096)
    noisy = x + sigma * torch.randn(4096)
    target = -(noisy - x) / sigma ** 2          # the score of p(x~ | x), easy to compute
    loss = ((a * noisy - target) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"DSM-learned slope a = {a.item():.4f}")

# Even though each target is noisy, its average given x~ is the true marginal score
x = torch.randn(2_000_000)
noisy = x + sigma * torch.randn_like(x)
target = -(noisy - x) / sigma ** 2
bins = torch.tensor([-1.5, -0.5, 0.5, 1.5])
print("\n  x~ near   mean DSM target   true score")
for b in bins:
    sel = (noisy - b).abs() < 0.05
    print(f"  {b.item():+.1f}      {target[sel].mean().item():+.4f}          {exact_slope * b.item():+.4f}")

closed-form score of the noisy marginal: s(x) = -0.8000 x


DSM-learned slope a = -0.8085

  x~ near   mean DSM target   true score
  -1.5      +1.1793          +1.2000
  -0.5      +0.3970          +0.4000
  +0.5      -0.4006          -0.4000
  +1.5      -1.2139          -1.2000
